# 1. Create a python file that webscrapes GDP by country and plots a stacked interactive bar plot using plotly. Stack countries within regions using the IMF numbers. Please include this in your ipython notebook and output your plot to an html file containing the plot.
## a. Create a python file that webscrapes GDP by country

In [1]:
import pandas as pd
import requests as rq
import bs4
from io import StringIO
import plotly.express as px
import numpy as np

In [ ]:
pd.set_option('display.max_rows', None)

#Nina from class helped debug/get through API 
url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15"}
page = rq.get(url, headers=headers)

page.text[0:99]
bs4page = bs4.BeautifulSoup(page.text,'html.parser')
table = bs4page.find('table', {'class': "wikitable"})
GDP_list = pd.read_html(StringIO(str(table)))
GDP = GDP_list[0]

print(GDP)


                    Country/Territory  IMF (2026)[1] World Bank (2024)[6]  \
0                               World      123584494            111326370   
1                       United States       31821293             28750956   
2                          China[n 1]       20650754             18743803   
3                             Germany        5328184              4685593   
4                               India        4515629              3909892   
5                               Japan        4463634              4027598   
6                      United Kingdom        4225639              3686033   
7                              France        3558562              3160443   
8                               Italy        2701544              2380825   
9                              Russia        2509421              2173836   
10                             Canada        2420837              2243637   
11                             Brazil        2292690              2185822   

 ## b. Plot a stacked interactive bar plot using plotly. Stack countries within regions using the IMF numbers. Please include this in your ipython notebook

In [3]:
import plotly.express as px

#Data cleaning recommendations by Gemini
clean_df = pd.DataFrame({
    'Country': GDP.iloc[:, 0],
    'IMF_GDP': GDP.iloc[:, 1]
})
clean_df['Country'] = clean_df['Country'].str.replace(r'\[.*?\]', '', regex=True).str.strip()
clean_df = clean_df[clean_df['Country'] != 'World']
clean_df['IMF_GDP'] = pd.to_numeric(clean_df['IMF_GDP'], errors='coerce')
clean_df = clean_df.dropna(subset=['IMF_GDP'])

#Assign continent
df_regions = px.data.gapminder()[['country', 'continent']].drop_duplicates()
merged_df = pd.merge(clean_df, df_regions, left_on='Country', right_on='country', how='left')
merged_df['continent'] = merged_df['continent'].fillna('Other/Unknown')

#Individual corrections
custom_continents = {
    'Russia': 'Europe', 'South Korea': 'Asia', 'United Arab Emirates': 'Asia',
    'Hong Kong': 'Asia', 'Kazakhstan': 'Asia', 'Qatar': 'Asia',
    'Ukraine': 'Europe', 'Slovakia': 'Europe', 'Uzbekistan': 'Asia',
    'Ivory Coast': 'Africa', 'Luxembourg': 'Europe', 'Lithuania': 'Europe',
    'Belarus': 'Europe', 'DR Congo': 'Africa', 'Azerbaijan': 'Asia',
    'Turkmenistan': 'Asia', 'Macau': 'Asia', 'Latvia': 'Europe',
    'Estonia': 'Europe', 'Cyprus': 'Europe', 'Georgia': 'Asia',
    'Papua New Guinea': 'Oceania', 'Malta': 'Europe', 'Armenia': 'Asia',
    'Guyana': 'Americas', 'Kyrgyzstan': 'Asia', 'Moldova': 'Europe',
    'North Macedonia': 'Europe', 'Tajikistan': 'Asia', 'Laos': 'Asia',
    'Yemen': 'Asia', 'Congo': 'Africa', 'Bahamas': 'Americas',
    'Brunei': 'Asia', 'Kosovo': 'Europe', 'Liechtenstein': 'Europe',
    'Maldives': 'Asia', 'Barbados': 'Americas', 'Fiji': 'Oceania',
    'South Sudan': 'Africa', 'Eswatini': 'Africa', 'Suriname': 'Americas',
    'Andorra': 'Europe', 'Aruba': 'Americas', 'Bhutan': 'Asia',
    'Belize': 'Americas', 'Cape Verde': 'Africa', 'Saint Lucia': 'Americas',
    'Antigua and Barbuda': 'Americas', 'San Marino': 'Europe',
    'Seychelles': 'Africa', 'Timor-Leste': 'Asia', 'Solomon Islands': 'Oceania',
    'Grenada': 'Americas', 'Samoa': 'Oceania',
    'Saint Vincent and the Grenadines': 'Americas',
    'Saint Kitts and Nevis': 'Americas', 'Vanuatu': 'Oceania',
    'São Tomé and Príncipe': 'Africa', 'Dominica': 'Americas',
    'Tonga': 'Oceania', 'Micronesia': 'Oceania', 'Palau': 'Oceania',
    'Kiribati': 'Oceania', 'Marshall Islands': 'Oceania', 'Nauru': 'Oceania',
    'Tuvalu': 'Oceania'
}

for country, continent in custom_continents.items():
    # This finds the specific country row, and overwrites the 'continent' column
    merged_df.loc[merged_df['Country'] == country, 'continent'] = continent
merged_df

#Make the Plotly
fig = px.bar(
    merged_df, 
    x='continent', 
    y='IMF_GDP', 
    color='Country',  # Stacks countries within regions
    title='Nominal GDP by Region and Country (IMF)',
    labels={'IMF_GDP': 'GDP (Nominal in Millions USD)', 'continent': 'UN Region'}
)
fig.update_layout(showlegend=False)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': ('Country=United States<br>UN Re' ... 'lions USD)=%{y}<extra></extra>'),
              'legendgroup': 'United States',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'United States',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['Americas'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'AAAA0N5YfkE=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('Country=China<br>UN Region=%{x' ... 'lions USD)=%{y}<extra></extra>'),
              'legendgroup': 'China',
              'marker': {'color': '#EF553B', 'pattern': {'shape': ''}},
              'name': 'China',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['Asia'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'AAAAILCxc0E=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('Country=Germany<br>UN Region=%' ... 'lions USD)=%{y}<extra></extra>'),
              'legendgroup': 'Germany',
              'marker': {'color': '#00cc96', 'pattern': {'shape': ''}},
              'name': 'Germany',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['Europe'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'AAAAAE5TVEE=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('Country=India<br>UN Region=%{x' ... 'lions USD)=%{y}<extra></extra>'),
              'legendgroup': 'India',
              'marker': {'color': '#ab63fa', 'pattern': {'shape': ''}},
              'name': 'India',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['Asia'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'AAAAQMs5UUE=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('Country=Japan<br>UN Region=%{x' ... 'lions USD)=%{y}<extra></extra>'),
              'legendgroup': 'Japan',
              'marker': {'color': '#FFA15A', 'pattern': {'shape': ''}},
              'name': 'Japan',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['Asia'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'AAAAgAQHUUE=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('Country=United Kingdom<br>UN R' ... 'lions USD)=%{y}<extra></extra>'),
              'legendgroup': 'United Kingdom',
              'marker': {'color': '#19d3f3', 'pattern': {'shape': ''}},
              'name': 'United Kingdom',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['Europe'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'AAAAwJkeUEE=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('Country=France<br>UN Region=%{' ... 'lions USD)=%{y}<extra></extra>'),
              'legendgroup': 'France',
              'marker': {'color': '#FF6692', 'pattern': {'shape': ''}},
              'name': 'France',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['Europe'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'AAAAAFEmS0E=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': ('Country=Italy<br>UN Region=%{x' ... 'lions USD)=%{y}<extra></extra>'),
 

## c. Output your plot to an html file containing the plot.

In [4]:
fig.write_html("stacked_bar.html")

# 2. Look at the chapter on interactive graphics and, specifically, the code to display a subject's MRICloud data as a sunburst plot. Do the following. Display this subject's data as a Sankey diagram. Display as many levels as you can (at least 3) for Type = 1, starting from the intracranial volume.

## a. Look at the chapter on interactive graphics and, specifically, the code to display a subject's MRICloud data as a sunburst plot.

In [5]:
#Download data
url = "https://raw.githubusercontent.com/smart-stats/ds4bio_book/main/book/assetts/kirby21AllLevels.csv"
response = rq.get(url)
subjectData = pd.read_csv(StringIO(response.text))
subjectData = subjectData.loc[(subjectData.type == 1) & (subjectData.level == 5)]
subjectData = subjectData[['roi', 'volume']]



## b. Display this subject's data as a Sankey diagram. Display as many levels as you can (at least 3) for Type = 1, starting from the intracranial volume.

In [6]:
#Multi level look up
url2 = "https://raw.githubusercontent.com/bcaffo/MRIcloudT1volumetrics/master/inst/extdata/multilevel_lookup_table.txt"
response2 = rq.get(url2)
multilevel_lookup = pd.read_csv(StringIO(response2.text), sep = "\t").drop(['Level5'], axis = 1)
multilevel_lookup = multilevel_lookup.rename(columns = {
    "modify"   : "roi",
    "modify.1" : "level4",
    "modify.2" : "level3",
    "modify.3" : "level2",
    "modify.4" : "level1"})
multilevel_lookup = multilevel_lookup[['roi', 'level4', 'level3', 'level2', 'level1']]
multilevel_lookup

#Merge data
subjectData_multilevel = pd.merge(subjectData, multilevel_lookup, on = "roi")
subjectData_multilevel = subjectData_multilevel.assign(icv = "ICV")
subjectData_multilevel = subjectData_multilevel.assign(comp = subjectData_multilevel.volume / np.sum(subjectData_multilevel.volume))


In [ ]:
#Create Sunburst
sunburst = px.sunburst(subjectData_multilevel, path=['icv', 'level1', 'level2', 'level3', 'level4', 'roi'],
                  values='comp', width=800, height=800)

#sunburst.write_html("sunburst.html")

In [7]:
#This code chunk is adapted from Gemini 
import plotly.graph_objects as go

# 1. Stack the levels using your finalized dataframe
links = pd.concat([
    subjectData_multilevel[['icv', 'level1', 'volume']].rename(columns={'icv':'source', 'level1':'target'}),
    subjectData_multilevel[['level1', 'level2', 'volume']].rename(columns={'level1':'source', 'level2':'target'}),
    subjectData_multilevel[['level2', 'level3', 'volume']].rename(columns={'level2':'source', 'level3':'target'}),
    subjectData_multilevel[['level3', 'level4', 'volume']].rename(columns={'level3':'source', 'level4':'target'})
])

# 2. Add up volumes for identical paths
links = links.groupby(['source', 'target'], as_index=False).sum()

# 3. Create a list of all unique brain region names
nodes = list(set(links['source']).union(set(links['target'])))

# 4. Build the Sankey chart
fig_sankey = go.Figure(go.Sankey(
    node = dict(label = nodes, pad=15, thickness=20),
    link = dict(
        source = [nodes.index(x) for x in links['source']],
        target = [nodes.index(x) for x in links['target']],
        value  = links['volume']
    )
))

fig_sankey.update_layout(title_text="Brain Volume Flow (ICV to Level 4)", height=800)

# 5. Save as index.html for your GitHub repository
fig_sankey.write_html("sankey.html")